# Logoloc: avaliação e consolidação (Colab)

Roda a avaliação Fluxo 1 / Fluxo 2 dos modelos já treinados e monta a tabela comparativa + gráfico IoU x classificação. Não treina nada.

**Antes de rodar**: `Ambiente de execução -> Alterar tipo de ambiente de execução -> GPU`.

**Antes da primeira execução**: suba o `logoloc_code.zip` mais recente pra Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/TCC II'
import os
assert os.path.exists(DRIVE_DIR), f'Pasta nao encontrada: {DRIVE_DIR}'
print('OK', os.listdir(DRIVE_DIR))

## 1. Código

In [ ]:
import zipfile

PROJECT_DIR = '/content/tcc'
CODE_ZIP = f'{DRIVE_DIR}/logoloc_code.zip'
assert os.path.exists(CODE_ZIP), f'Nao achei {CODE_ZIP}'

os.makedirs(PROJECT_DIR, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as zf:
    zf.extractall(PROJECT_DIR)

fixed = 0
for root, _dirs, files in os.walk(PROJECT_DIR):
    for name in files:
        if '\\' in name:
            old_path = os.path.join(root, name)
            new_path = os.path.join(root, *name.split('\\'))
            os.makedirs(os.path.dirname(new_path), exist_ok=True)
            os.replace(old_path, new_path)
            fixed += 1
if fixed:
    print(f'corrigidos {fixed} arquivos')

%cd $PROJECT_DIR
assert os.path.exists('scripts/06_evaluate_faster_rcnn.py')
print('OK')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA disponivel:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma')

## 2. Dataset bruto + checkpoints

In [ ]:
DATASET_ZIP = f'{DRIVE_DIR}/FlickrLogos-32_dataset_v2.zip'
assert os.path.exists(DATASET_ZIP), f'Nao achei {DATASET_ZIP}'

RAW_DIR = f'{PROJECT_DIR}/data/raw/FlickrLogos-32'
expected = f'{RAW_DIR}/dataset_v2/FlickrLogos-v2/classes'

if os.path.isdir(expected):
    print('OK, ja extraido em', RAW_DIR)
else:
    import zipfile
    os.makedirs(f'{RAW_DIR}/dataset_v2', exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        zf.extractall(f'{RAW_DIR}/dataset_v2')
    assert os.path.isdir(expected)
    print('OK, extraido em', RAW_DIR)

In [ ]:
import subprocess

def restore(name, dest):
    src = f'{DRIVE_DIR}/backup/{name}/'
    if os.path.isdir(src) and os.listdir(src):
        os.makedirs(dest, exist_ok=True)
        subprocess.run(['rsync', '-a', '--info=progress2', src, dest + '/'], check=True)
        print(f'restaurado: {name} -> {dest}')
    else:
        print(f'nada pra restaurar em {name}')

def backup(name, src):
    dest = f'{DRIVE_DIR}/backup/{name}/'
    os.makedirs(dest, exist_ok=True)
    subprocess.run(['rsync', '-a', '--info=progress2', src + '/', dest], check=True)
    print(f'backup: {src} -> {dest}')

restore('runs', 'outputs/runs')
restore('results', 'outputs/results')

## 3. Avaliação do Faster R-CNN

In [ ]:
FRCNN_RUNS = [
    ('frcnn_r50_v1', 'configs/faster_rcnn/resnet50.yaml'),
    ('frcnn_r101_v1', 'configs/faster_rcnn/resnet101.yaml'),
]

for run_name, model_config in FRCNN_RUNS:
    ckpt = f'outputs/runs/{run_name}/model_last.pt'
    if not os.path.exists(ckpt):
        print(f'pulando {run_name}: sem checkpoint em {ckpt}')
        continue
    print(f'=== avaliando {run_name} ===')
    subprocess.run([
        'python', 'scripts/06_evaluate_faster_rcnn.py',
        '--checkpoint', ckpt,
        '--model-config', model_config,
        '--run-name', run_name,
    ], check=True)

## 4. Sensibilidade ao erro de localizacao

Perturba as caixas anotadas ate niveis controlados de IoU e injeta cada uma na RoI head, sem passar pela RPN. Como as mesmas instancias aparecem em todos os niveis, mede diretamente quanto a classificacao piora conforme a caixa degrada.

In [ ]:
for run_name, model_config in FRCNN_RUNS:
    ckpt = f'outputs/runs/{run_name}/model_last.pt'
    if not os.path.exists(ckpt):
        print(f'pulando {run_name}: sem checkpoint em {ckpt}')
        continue
    print(f'=== sensibilidade {run_name} ===')
    subprocess.run([
        'python', 'scripts/10_localization_sensitivity.py',
        '--checkpoint', ckpt,
        '--model-config', model_config,
        '--run-name', run_name,
    ], check=True)

## 5. Consolidacao

In [ ]:
!python scripts/08_consolidate_results.py

In [ ]:
!python scripts/09_generate_report.py

In [ ]:
backup('results', 'outputs/results')

## 6. Ver os graficos

In [ ]:
from IPython.display import Image, display

display(Image(filename='outputs/results/iou_vs_classification.png'))
for run_name, _ in FRCNN_RUNS:
    caminho = f'outputs/results/{run_name}/localization_sensitivity.png'
    if os.path.exists(caminho):
        display(Image(filename=caminho))

In [ ]:
from google.colab import files
files.download('outputs/results/relatorio.md')